# 01 — Data Inspection (M2)

Inspect **controlled** (PlantVillage) vs **field** (CD&S) corn images side by side.

Checks: class distribution, derived `leaf_id` grouping, sample grids, image-size stats, corrupt-file scan.

Run the download scripts first:
- `python scripts/download_plantvillage.py`
- `python scripts/download_cds.py --verify`  (after manually placing CD&S in `data/raw/cds/`)

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.maize_detection.labels import CDS_TO_CANONICAL, CANONICAL_LABELS

PV_DIR = PROJECT_ROOT / 'data' / 'raw' / 'plantvillage'
PV_MANIFEST = PROJECT_ROOT / 'data' / 'processed' / 'plantvillage_manifest.csv'
CDS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'cds'
IMG_SUFFIXES = {'.jpg', '.jpeg', '.png'}
print('PlantVillage dir exists:', PV_DIR.exists())
print('Manifest exists:        ', PV_MANIFEST.exists())
print('CD&S dir exists:        ', CDS_DIR.exists())

## 1. PlantVillage (controlled) — manifest + class distribution

In [ ]:
import csv

pv_rows = []
if PV_MANIFEST.exists():
    with open(PV_MANIFEST, encoding='utf-8') as f:
        pv_rows = list(csv.DictReader(f))
    print(f'Manifest rows: {len(pv_rows):,}')
    print('Columns:', list(pv_rows[0].keys()) if pv_rows else '(empty)')
else:
    print('Manifest not found — run scripts/download_plantvillage.py first.')

pv_counts = Counter(r['canonical_label'] for r in pv_rows)
for label in CANONICAL_LABELS:
    print(f'  {label:<22} {pv_counts.get(label, 0):>5}')

In [ ]:
if pv_rows:
    fig, ax = plt.subplots(figsize=(7, 3.5))
    labels = CANONICAL_LABELS
    vals = [pv_counts.get(l, 0) for l in labels]
    ax.bar(labels, vals, color='#4c72b0')
    ax.set_title('PlantVillage corn — images per class (controlled)')
    ax.set_ylabel('images')
    plt.xticks(rotation=20, ha='right')
    for i, v in enumerate(vals):
        ax.text(i, v + 5, str(v), ha='center')
    plt.tight_layout(); plt.show()

## 2. Derived `leaf_id` grouping (CARDINAL INVARIANT)

Corn has no official leaf_id; it is derived from filenames. Verify it is present, populated, and that multiple images share leaves (otherwise grouping is meaningless). **common_rust is expected to show no grouping** — a documented residual risk.

In [ ]:
if pv_rows:
    assert all(r['leaf_id'] for r in pv_rows), 'Empty leaf_id found!'
    print(f"{'class':<22}{'images':>8}{'leaves':>8}{'imgs/leaf':>11}")
    for label in CANONICAL_LABELS:
        imgs = [r for r in pv_rows if r['canonical_label'] == label]
        leaves = {r['leaf_id'] for r in imgs}
        ratio = len(imgs) / len(leaves) if leaves else 0
        print(f'{label:<22}{len(imgs):>8}{len(leaves):>8}{ratio:>11.2f}')
    total_leaves = len({r['leaf_id'] for r in pv_rows})
    print(f"{'TOTAL':<22}{len(pv_rows):>8}{total_leaves:>8}")
    rust_imgs = [r for r in pv_rows if r['canonical_label'] == 'common_rust']
    rust_leaves = len({r['leaf_id'] for r in rust_imgs})
    if rust_imgs and rust_leaves == len(rust_imgs):
        print('\nNOTE: common_rust has no detectable grouping (each image = own leaf).')

## 3. Sample grids

In [ ]:
def show_grid(image_paths, title, ncols=5):
    """Render up to ncols images in a row."""
    paths = image_paths[:ncols]
    if not paths:
        print(f'{title}: no images'); return
    fig, axes = plt.subplots(1, len(paths), figsize=(2.2 * len(paths), 2.6))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        try:
            ax.imshow(Image.open(p).convert('RGB'))
        except Exception as e:
            ax.text(0.5, 0.5, f'err\n{e}', ha='center', fontsize=6)
        ax.axis('off')
    fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()

def list_images(folder):
    if not Path(folder).exists():
        return []
    return sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in IMG_SUFFIXES)

In [ ]:
# PlantVillage (controlled) — one row per class
for label in CANONICAL_LABELS:
    show_grid(list_images(PV_DIR / label), f'PlantVillage / {label} (controlled)')

## 4. CD&S (field) — counts and samples

Field images: NLB and GLS map to canonical labels; **NLS is excluded** (no canonical match).

In [ ]:
EXPECTED_CDS = {'NLB': 511, 'GLS': 524, 'NLS': 562}
print(f"{'CD&S class':<8}{'found':>7}{'expected':>10}  canonical")
for cls, exp in EXPECTED_CDS.items():
    n = len(list_images(CDS_DIR / cls))
    canon = CDS_TO_CANONICAL[cls] or 'EXCLUDED'
    print(f'{cls:<8}{n:>7}{exp:>10}  {canon}')

In [ ]:
for cls in ('NLB', 'GLS', 'NLS'):
    canon = CDS_TO_CANONICAL[cls] or 'EXCLUDED'
    show_grid(list_images(CDS_DIR / cls), f'CD&S / {cls} -> {canon} (field)')

## 5. Controlled vs field — the domain gap, visually

The two overlapping classes (NLB, GLS) side by side. This is the gap the project measures.

In [ ]:
pairs = [('gray_leaf_spot', 'GLS'), ('northern_leaf_blight', 'NLB')]
for canon, cds_cls in pairs:
    show_grid(list_images(PV_DIR / canon), f'CONTROLLED  {canon}', ncols=4)
    show_grid(list_images(CDS_DIR / cds_cls), f'FIELD       {canon} (CD&S {cds_cls})', ncols=4)

## 6. Image-size stats + corrupt-file scan

In [ ]:
def scan(folder_map, name):
    sizes, corrupt, total = [], [], 0
    for folder in folder_map:
        for p in list_images(folder):
            total += 1
            try:
                with Image.open(p) as im:
                    im.verify()
                with Image.open(p) as im:
                    sizes.append(im.size)
            except Exception as e:
                corrupt.append((p, e))
    print(f'\n=== {name} ===')
    print(f'  images scanned: {total:,}')
    print(f'  corrupt:        {len(corrupt)}')
    for p, e in corrupt[:5]:
        print(f'    {p.name}: {e}')
    if sizes:
        ws = [s[0] for s in sizes]; hs = [s[1] for s in sizes]
        print(f'  width  min/max: {min(ws)} / {max(ws)}')
        print(f'  height min/max: {min(hs)} / {max(hs)}')
        print(f'  most common size: {Counter(sizes).most_common(1)[0]}')

scan([PV_DIR / l for l in CANONICAL_LABELS], 'PlantVillage (controlled)')
scan([CDS_DIR / c for c in ('NLB', 'GLS', 'NLS')], 'CD&S (field)')

## Checkpoint

Before M3, confirm: 4 PlantVillage classes present with expected counts; `leaf_id` populated and grouping plausible (healthy/NLB/GLS group, common_rust does not); CD&S counts ~511/524/562; controlled images are clean single leaves while field images have natural backgrounds. **Pause here and review with the human.**